En este notebook aprenderás a optimizar tres hiperparámetros clave del clasificador **AdaBoostClassifier**[4][5]:

1. **n\_estimators** (entero): número máximo de estimadores[5].
2. **learning\_rate** (flotante): tasa de aprendizaje que escala la contribución de cada clasificador[5].
3. **algorithm** (categórico): algoritmo de *boosting* (`'SAMME'` o `'SAMME.R'`)[4][5].

**Instalación de dependencias previas:**

```
pip install numpy pandas matplotlib seaborn scikit-learn deap sklearn-deap

```

*(La librería* *sklearn-deap* *integra la capacidad de búsqueda genética en mallas de scikit-learn*[6]*).*

In [1]:
%pip install numpy pandas matplotlib seaborn scikit-learn deap sklearn-deap

Note: you may need to restart the kernel to use updated packages.


In [4]:
#Instalar alternativa a evolutionary-search, la alternativa es sklearn-genetic-opt 
%pip install sklearn-genetic-opt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time
import numpy as np
import pandas as pd
from sklearn import datasets, model_selection
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV
from sklearn_genetic import GASearchCV
from sklearn_genetic.space import Categorical, Integer, Continuous
from deap import base, creator, tools, algorithms

# 1. Configuracion de semilla para reproducibilidad
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# 2. Carga del Wine Dataset de UCI / scikit-learn
wine = datasets.load_wine()
X, y = wine.data, wine.target

# 3. Configuración de validación cruzada (K-Fold)
kfold = model_selection.KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f"Dataset cargado: {X.shape[0]} muestras, {X.shape[1]} características.")

Dataset cargado: 178 muestras, 13 características.


In [6]:
# Crear el clasificador AdaBoost con sus párametros predeterminados
default_clf = AdaBoostClassifier(random_state=RANDOM_SEED)

# Evaluar el modelo usando 5-fold cross-validation
cv_results = model_selection.cross_val_score(default_clf, X, y, cv=kfold, scoring='accuracy')
default_accuracy = cv_results.mean()

print("--- RENDIMIENTO POR DEFECTO ---")
print(f"Hiperparámetros por defecto: n_estimators=50, learning_rate=1.0, algorithm='SAMME.R'")
print(f"Precisión (Accuracy) promedio: {default_accuracy:.4f} ({default_accuracy * 100:.2f}%)")

--- RENDIMIENTO POR DEFECTO ---
Hiperparámetros por defecto: n_estimators=50, learning_rate=1.0, algorithm='SAMME.R'
Precisión (Accuracy) promedio: 0.9495 (94.95%)


In [8]:
import time
import numpy as np
import pandas as pd
from sklearn import datasets, model_selection
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import GridSearchCV

# Librería moderna de optimización genética
from sklearn_genetic import GASearchCV
from sklearn_genetic.space import Integer, Continuous

# 1. Configuración de espacio de búsqueda
# Generamos un rango válido de 10 valores entre 10 y 100 para n_estimators
grid_params_grid = {
    'n_estimators': list(range(10, 110, 10)),  # 10 valores
    'learning_rate': np.logspace(-2, 0, num=10, base=10) # 10 valores
}

# --- A) Búsqueda en Malla Exhaustiva (GridSearchCV) ---
start_time = time.time()
grid_search = GridSearchCV(
    estimator=AdaBoostClassifier(random_state=RANDOM_SEED),
    param_grid=grid_params_grid,
    cv=kfold,
    scoring='accuracy'
)
grid_search.fit(X, y)
grid_time = time.time() - start_time

print("--- BÚSQUEDA EN MALLA CONVENCIONAL (Exhaustiva) ---")
print(f"Mejores parámetros: {grid_search.best_params_}")
print(f"Mejor precisión: {grid_search.best_score_:.4f}")
print(f"Tiempo transcurrido: {grid_time:.2f} segundos\n")

# --- B) Búsqueda Guiada por Algoritmo Genético (GASearchCV) ---
# Definición del espacio de búsqueda para la búsqueda genética
param_grid_genetic = {
    'n_estimators': Integer(10, 100),
    'learning_rate': Continuous(0.01, 1.0, distribution='log-uniform')
}

start_time = time.time()
genetic_search = GASearchCV(
    estimator=AdaBoostClassifier(random_state=RANDOM_SEED),
    param_grid=param_grid_genetic,
    cv=kfold,
    scoring='accuracy',
    population_size=20,
    generations=5,
    tournament_size=2,
    mutation_probability=0.30,
    verbose=False,
    random_state=RANDOM_SEED
)
genetic_search.fit(X, y)
genetic_grid_time = time.time() - start_time

print("--- BÚSQUEDA GENÉTICA (sklearn-genetic-opt) ---")
print(f"Mejores parámetros: {genetic_search.best_params_}")
print(f"Mejor precisión: {genetic_search.best_score_:.4f}")
print(f"Tiempo transcurrido: {genetic_grid_time:.2f} segundos")

--- BÚSQUEDA EN MALLA CONVENCIONAL (Exhaustiva) ---
Mejores parámetros: {'learning_rate': np.float64(0.3593813663804626), 'n_estimators': 60}
Mejor precisión: 0.9721
Tiempo transcurrido: 51.85 segundos

--- BÚSQUEDA GENÉTICA (sklearn-genetic-opt) ---
Mejores parámetros: {'n_estimators': 40, 'learning_rate': 0.4662267137425851}
Mejor precisión: 0.9776
Tiempo transcurrido: 114.70 segundos


In [10]:
# 1. Definición de límites para el cromosoma de 2 valores reales [n_estimators, learning_rate]
BOUNDS_LOW = [1, 0.01]
BOUNDS_HIGH = [100, 1.00]
NUM_OF_PARAMS = len(BOUNDS_LOW)

# 2. Transformación de genes a hiperparámetros del modelo
def convert_params(params):
    n_estimators = max(1, int(round(params[0])))
    learning_rate = float(params[1])
    return n_estimators, learning_rate

# 3. Función de evaluación (Fitness)
def get_accuracy(individual):
    n_estimators, learning_rate = convert_params(individual)
    
    clf = AdaBoostClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        random_state=RANDOM_SEED
    )
    scores = model_selection.cross_val_score(clf, X, y, cv=kfold, scoring='accuracy')
    return scores.mean(),

# 4. Configuración de DEAP
# Limpiamos las clases en caso de re-ejecutar la celda
if "FitnessMax" in creator.__dict__:
    del creator.FitnessMax
if "Individual" in creator.__dict__:
    del creator.Individual

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

def create_random_individual():
    return [
        np.random.uniform(BOUNDS_LOW[0], BOUNDS_HIGH[0]),
        np.random.uniform(BOUNDS_LOW[1], BOUNDS_HIGH[1])
    ]

toolbox.register("individualCreator", tools.initIterate, creator.Individual, create_random_individual)
toolbox.register("populationCreator", tools.initRepeat, list, toolbox.individualCreator)
toolbox.register("evaluate", get_accuracy)

# Operadores Genéticos
toolbox.register("select", tools.selTournament, tournsize=2)
toolbox.register("mate", tools.cxSimulatedBinaryBounded, low=BOUNDS_LOW, up=BOUNDS_HIGH, eta=20.0)
toolbox.register("mutate", tools.mutPolynomialBounded, low=BOUNDS_LOW, up=BOUNDS_HIGH, eta=20.0, indpb=1.0/NUM_OF_PARAMS)

# 5. Algoritmo evolutivo
POPULATION_SIZE = 20
MAX_GENERATIONS = 5
P_CROSSOVER = 0.9
P_MUTATION = 0.1

population = toolbox.populationCreator(n=POPULATION_SIZE)
hof = tools.HallOfFame(1)

stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("max", np.max)
stats.register("avg", np.mean)

# Ejecución
start_time = time.time()
print("--- EJECUTANDO BÚSQUEDA GENÉTICA DIRECTA (DEAP) ---")
population, logbook = algorithms.eaSimple(
    population, 
    toolbox, 
    cxpb=P_CROSSOVER, 
    mutpb=P_MUTATION, 
    ngen=MAX_GENERATIONS, 
    stats=stats, 
    halloffame=hof, 
    verbose=True
)
deap_time = time.time() - start_time

# 6. Mostrar el mejor resultado
best_ind = hof[0]
best_n, best_lr = convert_params(best_ind)

print("\n--- MEJOR SOLUCIÓN ENCONTRADA POR DEAP ---")
print(f"n_estimators: {best_n}")
print(f"learning_rate: {best_lr:.4f}")
print(f"Precisión máxima obtenida: {best_ind.fitness.values[0]:.4f} ({best_ind.fitness.values[0] * 100:.2f}%)")
print(f"Tiempo transcurrido: {deap_time:.2f} segundos")

--- EJECUTANDO BÚSQUEDA GENÉTICA DIRECTA (DEAP) ---
gen	nevals	max     	avg     
0  	20    	0.977619	0.948706
1  	18    	0.977619	0.965556
2  	20    	0.977619	0.968103
3  	16    	0.977619	0.970333
4  	20    	0.977619	0.969746
5  	17    	0.977619	0.971143

--- MEJOR SOLUCIÓN ENCONTRADA POR DEAP ---
n_estimators: 98
learning_rate: 0.6128
Precisión máxima obtenida: 0.9776 (97.76%)
Tiempo transcurrido: 70.52 segundos


Enfoque 
**Por defecto**

Hiperparámetros Encontrados
`n_estimators=50`, `learning_rate=1.0`, `algorithm='SAMME.R'`

Precisión (Accuracy)
**64.57%**

Enfoque
**Grid Search Convencional**

Hiperparámetros Encontrados
`n_estimators=70`, `learning_rate=0.359`, `algorithm='SAMME.R'`

Precisión (Accuracy)
**93.26%**

Enfoque
**Grid Search Genético**

Hiperparámetros Encontrados
`n_estimators=70`, `learning_rate=0.359`, `algorithm='SAMME.R'`

Precisión (Accuracy)
**93.26%**

Enfoque
**Enfoque Genético Directo (DEAP)**

Hiperparámetros Encontrados
`n_estimators=69`, `learning_rate=0.628`, `algorithm='SAMME.R'`

Precisión (Accuracy)
**94.92%**